In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, count, sum

spark = SparkSession.builder \
    .appName("titanic-gold") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://titanic-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [2]:
df = spark.read.format("delta") \
    .load("s3a://titanic/lakehouse/silver/titanic")

In [3]:
df_gold = df.groupBy("Pclass", "Sex") \
    .agg(
        count("*").alias("total"),
        sum(col("Survived").cast("int")).alias("sobreviventes"),
        round(sum(col("Survived").cast("int")) / count("*") * 100, 1).alias("taxa_sobrevivencia_%")
    ) \
    .orderBy("Pclass", "Sex")

df_gold.show()

+------+------+-----+-------------+--------------------+
|Pclass|   Sex|total|sobreviventes|taxa_sobrevivencia_%|
+------+------+-----+-------------+--------------------+
|     1|female|   50|           50|               100.0|
|     1|  male|   57|            0|                 0.0|
|     2|female|   30|           30|               100.0|
|     2|  male|   63|            0|                 0.0|
|     3|female|   72|           72|               100.0|
|     3|  male|  146|            0|                 0.0|
+------+------+-----+-------------+--------------------+



In [4]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://titanic/lakehouse/gold/titanic")

print("Gold gravado com sucesso")

Gold gravado com sucesso
